# 🚗 Trip Intelligence Engine

This notebook generates intelligent trip planning insights for EV drivers by:
* Analyzing optimal routes based on real-time hazard data
* Identifying charging station requirements along routes
* Calculating trip risk scores and estimated travel times
* Providing time-based recommendations for safer travel

**Output Tables:**
* `mobility_ai.gold.trip_routes_optimal` - Optimal route analysis
* `mobility_ai.gold.trip_charging_requirements` - Charging planning data
* `mobility_ai.gold.trip_recommendations` - Smart trip recommendations

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime
import math

# Configuration
TRAFFIC_HAZARDS_TABLE = "mobility_ai.silver.traffic_hazards"
EV_STATIONS_TABLE = "mobility_ai.silver.ev_charging_stations"
CONGESTION_FORECAST_TABLE = "mobility_ai.gold.congestion_forecast_hourly"
CONGESTION_LOCATION_TABLE = "mobility_ai.gold.congestion_forecast_location"

# Output tables
TRIP_ROUTES_TABLE = "mobility_ai.gold.trip_routes_optimal"
TRIP_CHARGING_TABLE = "mobility_ai.gold.trip_charging_requirements"
TRIP_RECOMMENDATIONS_TABLE = "mobility_ai.gold.trip_recommendations"

# Trip planning parameters
AVG_EV_RANGE_KM = 350  # Average EV range on full charge
AVG_SPEED_KMH = 60  # Average urban driving speed
CHARGING_BUFFER_KM = 50  # Reserve range before charging needed
SAFE_TRIP_RISK_SCORE = 40  # Risk score threshold for safe travel

print("\n" + "═" * 80)
print("🚗 TRIP INTELLIGENCE ENGINE - CONFIGURATION")
print("═" * 80)
print("\n📊 Source Tables:")
print(f"   • Traffic Hazards:     {TRAFFIC_HAZARDS_TABLE}")
print(f"   • EV Stations:         {EV_STATIONS_TABLE}")
print(f"   • Congestion Forecast: {CONGESTION_FORECAST_TABLE}")
print(f"   • Location Risk:       {CONGESTION_LOCATION_TABLE}")
print("\n🎯 Output Tables:")
print(f"   • Optimal Routes:      {TRIP_ROUTES_TABLE}")
print(f"   • Charging Needs:      {TRIP_CHARGING_TABLE}")
print(f"   • Recommendations:     {TRIP_RECOMMENDATIONS_TABLE}")
print("\n⚙️ Trip Parameters:")
print(f"   • Average EV Range:    {AVG_EV_RANGE_KM} km")
print(f"   • Average Speed:       {AVG_SPEED_KMH} km/h")
print(f"   • Charging Buffer:     {CHARGING_BUFFER_KM} km")
print(f"   • Safe Risk Threshold: {SAFE_TRIP_RISK_SCORE}/100")
print("═" * 80 + "\n")

In [0]:
# Load source data
traffic_hazards = spark.table(TRAFFIC_HAZARDS_TABLE)
ev_stations = spark.table(EV_STATIONS_TABLE)
congestion_forecast = spark.table(CONGESTION_FORECAST_TABLE)
location_risk = spark.table(CONGESTION_LOCATION_TABLE)

print("\n" + "═" * 80)
print("📊 DATA LOADING")
print("═" * 80)
print("\n✅ Source Tables Loaded:")
print(f"   • Traffic Hazards:      {traffic_hazards.count():,}")
print(f"   • Active Hazards:       {traffic_hazards.filter(F.col('is_active')).count():,}")
print(f"   • EV Charging Stations: {ev_stations.count():,}")
print(f"   • Congestion Periods:   {congestion_forecast.count():,}")
print(f"   • Risk Locations:       {location_risk.count():,}")
print("\n" + "─" * 80)
print("\n📋 Key Metrics:")
print(f"   • High-Risk Locations:  {location_risk.filter(F.col('risk_level') == 'High Risk').count()}")
print(f"   • Fast Chargers:        {ev_stations.filter(F.col('charging_speed') == 'Fast').count()}")
print(f"   • Critical Periods:     {congestion_forecast.filter(F.col('congestion_level') == 'Critical').count()}")
print("═" * 80 + "\n")

In [0]:
# Create location-based trip analysis
# Analyze hazard density and risk by location
trip_routes = traffic_hazards.filter(
    F.col("has_valid_location") & F.col("location").isNotNull()
).groupBy("location").agg(
    # Hazard statistics
    F.count("*").alias("total_hazards"),
    F.sum(F.when(F.col("is_active"), 1).otherwise(0)).alias("active_hazards"),
    F.sum(F.when(F.col("severity") == "Major", 1).otherwise(0)).alias("major_hazards"),
    F.sum(F.when(F.col("severity") == "Moderate", 1).otherwise(0)).alias("moderate_hazards"),
    
    # Location coordinates (center point)
    F.avg("latitude").alias("center_lat"),
    F.avg("longitude").alias("center_lon"),
    
    # Hazard type breakdown
    F.sum(F.when(F.col("main_category") == "Scheduled Roadwork", 1).otherwise(0)).alias("roadwork_count"),
    F.sum(F.when(F.col("main_category") == "Hazard", 1).otherwise(0)).alias("incident_count"),
    F.sum(F.when(F.col("main_category") == "Flooding", 1).otherwise(0)).alias("flood_count"),
    F.sum(F.when(F.col("main_category") == "Special Event", 1).otherwise(0)).alias("event_count"),
    
    # Timing
    F.min("start_time_ts").alias("earliest_hazard_time"),
    F.max("end_time_ts").alias("latest_hazard_time")
)

# Calculate route risk score (0-100)
trip_routes = trip_routes.withColumn(
    "route_risk_score",
    F.least(
        F.lit(100),
        (
            # Base hazard density (max 30 points)
            F.least(F.lit(30), F.col("total_hazards") * 0.5) +
            
            # Major hazard penalty (max 35 points)
            (F.col("major_hazards") * 12) +
            
            # Active hazards (max 25 points)
            F.least(F.lit(25), F.col("active_hazards") * 1.5) +
            
            # Flooding critical factor (max 10 points)
            (F.col("flood_count") * 10)
        )
    ).cast("int")
)

# Categorize route safety
trip_routes = trip_routes.withColumn(
    "route_safety_rating",
    F.when(F.col("route_risk_score") >= 70, "Avoid - High Risk")
     .when(F.col("route_risk_score") >= 50, "Caution - Moderate Risk")
     .when(F.col("route_risk_score") >= 25, "Acceptable - Low Risk")
     .otherwise("Recommended - Minimal Risk")
)

# Identify primary hazard type
trip_routes = trip_routes.withColumn(
    "primary_hazard_type",
    F.when(F.col("flood_count") > 0, "Flooding - Road Closures Likely")
     .when(
        (F.col("roadwork_count") >= F.col("incident_count")) & 
        (F.col("roadwork_count") >= F.col("event_count")),
        "Roadwork - Expect Delays"
    ).when(
        (F.col("incident_count") >= F.col("roadwork_count")) & 
        (F.col("incident_count") >= F.col("event_count")),
        "Traffic Incidents"
    ).when(
        F.col("event_count") > 0,
        "Special Event - Heavy Traffic"
    ).otherwise("Mixed Hazards")
)

# Estimate average delay (minutes per hazard)
trip_routes = trip_routes.withColumn(
    "estimated_delay_minutes",
    F.when(F.col("route_safety_rating") == "Avoid - High Risk", F.col("active_hazards") * 15)
     .when(F.col("route_safety_rating") == "Caution - Moderate Risk", F.col("active_hazards") * 8)
     .when(F.col("route_safety_rating") == "Acceptable - Low Risk", F.col("active_hazards") * 3)
     .otherwise(F.col("active_hazards") * 1)
)

trip_routes = trip_routes.withColumn("analysis_timestamp", F.current_timestamp())

print("\n" + "═" * 80)
print("🗺️  ROUTE INTELLIGENCE ANALYSIS")
print("═" * 80)
print(f"\n📍 Analyzed {trip_routes.count():,} route locations")

# Safety distribution
print("\n🚦 Route Safety Distribution:")
for row in trip_routes.groupBy("route_safety_rating").count().orderBy("count", ascending=False).collect():
    print(f"   • {row['route_safety_rating']:30s}: {row['count']:3d} locations")

print("\n" + "─" * 80)
print("\n⚠️  Top 5 Routes to Avoid:")
trip_routes.orderBy("route_risk_score", ascending=False).select(
    "location",
    "route_risk_score",
    "route_safety_rating",
    "active_hazards",
    "primary_hazard_type",
    "estimated_delay_minutes"
).show(5, truncate=False)

print("✅ Top 5 Safest Routes:")
trip_routes.orderBy("route_risk_score").select(
    "location",
    "route_risk_score",
    "route_safety_rating",
    "active_hazards",
    "primary_hazard_type"
).show(5, truncate=False)

print("═" * 80 + "\n")

In [0]:
# Calculate charging requirements for different trip distances
# Create a grid of common trip distances
trip_distances = [(25,), (50,), (75,), (100,), (150,), (200,), (250,), (300,), (350,), (400,), (500,)]
trip_distances_df = spark.createDataFrame(trip_distances, ["trip_distance_km"])

# Calculate charging needs
charging_requirements = trip_distances_df.withColumn(
    "charging_stops_needed",
    F.ceil((F.col("trip_distance_km") - AVG_EV_RANGE_KM + CHARGING_BUFFER_KM) / (AVG_EV_RANGE_KM - CHARGING_BUFFER_KM))
).withColumn(
    "charging_stops_needed",
    F.when(F.col("charging_stops_needed") < 0, 0).otherwise(F.col("charging_stops_needed"))
)

# Calculate trip time estimates
charging_requirements = charging_requirements.withColumn(
    "base_driving_time_hours",
    (F.col("trip_distance_km") / AVG_SPEED_KMH)
).withColumn(
    "charging_time_hours",
    F.col("charging_stops_needed") * 0.5  # 30 minutes per fast charge
).withColumn(
    "total_trip_time_hours",
    F.col("base_driving_time_hours") + F.col("charging_time_hours")
)

# Add recommended charger types
charging_requirements = charging_requirements.withColumn(
    "recommended_charger_type",
    F.when(F.col("charging_stops_needed") > 0, "Fast Charger (50+ kW)")
     .otherwise("Any Charger")
)

# Calculate minimum battery level at start
charging_requirements = charging_requirements.withColumn(
    "minimum_starting_battery_pct",
    F.when(
        F.col("trip_distance_km") <= AVG_EV_RANGE_KM,
        F.ceil((F.col("trip_distance_km") + CHARGING_BUFFER_KM) / AVG_EV_RANGE_KM * 100)
    ).otherwise(100)
)

# Trip feasibility
charging_requirements = charging_requirements.withColumn(
    "trip_feasibility",
    F.when(F.col("trip_distance_km") <= AVG_EV_RANGE_KM, "Easy - No charging needed")
     .when(F.col("charging_stops_needed") <= 1, "Moderate - One charging stop")
     .when(F.col("charging_stops_needed") <= 3, "Complex - Multiple stops needed")
     .otherwise("Challenging - Plan carefully")
)

# Add nearby charger availability (join with station count)
charger_availability = ev_stations.groupBy().agg(
    F.count("*").alias("total_stations"),
    F.sum("number_of_plugs").alias("total_plugs"),
    F.sum(F.when(F.col("charging_speed") == "Fast", 1).otherwise(0)).alias("fast_charger_stations")
)

charging_requirements = charging_requirements.crossJoin(charger_availability)

# Calculate charger density score
charging_requirements = charging_requirements.withColumn(
    "charger_availability_score",
    F.when(F.col("fast_charger_stations") >= 100, "Excellent")
     .when(F.col("fast_charger_stations") >= 50, "Good")
     .when(F.col("fast_charger_stations") >= 20, "Moderate")
     .otherwise("Limited")
)

charging_requirements = charging_requirements.withColumn("calculated_at", F.current_timestamp())

print("\n" + "═" * 80)
print("🔋 CHARGING REQUIREMENTS ANALYSIS")
print("═" * 80)
print(f"\n📊 Analyzed {charging_requirements.count()} common trip distances")
print(f"\n⚡ Available Charging Infrastructure:")
charger_stats = charger_availability.first()
print(f"   • Total Stations:      {charger_stats['total_stations']:,}")
print(f"   • Total Plugs:         {int(charger_stats['total_plugs']):,}")
print(f"   • Fast Chargers:       {charger_stats['fast_charger_stations']:,}")
print(f"   • Network Status:      {charging_requirements.first()['charger_availability_score']}")

print("\n" + "─" * 80)
print("\n🗺️  Trip Planning Guide:")
charging_requirements.select(
    "trip_distance_km",
    "charging_stops_needed",
    F.round("total_trip_time_hours", 1).alias("total_hours"),
    "minimum_starting_battery_pct",
    "trip_feasibility"
).show(11, truncate=False)

print("═" * 80 + "\n")

In [0]:
# Generate smart trip recommendations by combining route risk and time
# Join route analysis with congestion forecast
trip_recommendations = congestion_forecast.select(
    "hour_of_day",
    "day_name",
    "congestion_score",
    "congestion_level",
    "total_hazards",
    "active_hazards",
    "is_peak_hour"
).distinct()

# Calculate overall trip suitability score (0-100, higher is better)
trip_recommendations = trip_recommendations.withColumn(
    "trip_suitability_score",
    F.greatest(
        F.lit(0),
        100 - F.col("congestion_score")
    ).cast("int")
)

# Categorize trip windows
trip_recommendations = trip_recommendations.withColumn(
    "trip_window_category",
    F.when(F.col("trip_suitability_score") >= 90, "Optimal")
     .when(F.col("trip_suitability_score") >= 70, "Good")
     .when(F.col("trip_suitability_score") >= 50, "Fair")
     .when(F.col("trip_suitability_score") >= 30, "Poor")
     .otherwise("Avoid")
)

# Add time period classification
trip_recommendations = trip_recommendations.withColumn(
    "time_period",
    F.when(F.col("hour_of_day").between(0, 5), "Late Night")
     .when(F.col("hour_of_day").between(6, 9), "Morning Rush")
     .when(F.col("hour_of_day").between(10, 11), "Late Morning")
     .when(F.col("hour_of_day").between(12, 14), "Midday")
     .when(F.col("hour_of_day").between(15, 17), "Afternoon")
     .when(F.col("hour_of_day").between(18, 20), "Evening Rush")
     .otherwise("Night")
)

# Generate specific recommendations
trip_recommendations = trip_recommendations.withColumn(
    "trip_advice",
    F.when(
        F.col("trip_window_category") == "Optimal",
        "Excellent time to travel - minimal congestion and hazards"
    ).when(
        F.col("trip_window_category") == "Good",
        "Good travel conditions - some minor delays possible"
    ).when(
        F.col("trip_window_category") == "Fair",
        "Moderate traffic expected - allow extra time"
    ).when(
        F.col("trip_window_category") == "Poor",
        "Heavy traffic likely - consider alternative time"
    ).otherwise(
        "Not recommended - severe congestion and/or hazards"
    )
)

# Add charging station availability by time
trip_recommendations = trip_recommendations.withColumn(
    "charging_availability",
    F.when(F.col("hour_of_day").between(0, 6), "24/7 stations only")
     .otherwise("All stations available")
)

# Estimate additional travel time due to congestion
trip_recommendations = trip_recommendations.withColumn(
    "congestion_delay_factor",
    F.when(F.col("congestion_level") == "Critical", 1.5)
     .when(F.col("congestion_level") == "High", 1.3)
     .when(F.col("congestion_level") == "Moderate", 1.15)
     .when(F.col("congestion_level") == "Low", 1.05)
     .otherwise(1.0)
)

# Calculate priority ranking
window_spec = Window.orderBy(F.col("trip_suitability_score").desc(), F.col("hour_of_day"))
trip_recommendations = trip_recommendations.withColumn(
    "priority_rank",
    F.row_number().over(window_spec)
)

trip_recommendations = trip_recommendations.withColumn("generated_at", F.current_timestamp())

print("\n" + "═" * 80)
print("💡 SMART TRIP RECOMMENDATIONS")
print("═" * 80)
print(f"\n📊 Generated {trip_recommendations.count()} time-based recommendations")

# Distribution by category
print("\n🚦 Travel Window Distribution:")
for row in trip_recommendations.groupBy("trip_window_category").count().orderBy("count", ascending=False).collect():
    total = trip_recommendations.count()
    pct = (row['count'] / total) * 100
    print(f"   • {row['trip_window_category']:10s}: {row['count']:3d} time slots ({pct:5.1f}%)")

print("\n" + "─" * 80)
print("\n⭐ Top 10 Best Times to Travel:")
trip_recommendations.orderBy("priority_rank").select(
    "hour_of_day",
    "time_period",
    "trip_suitability_score",
    "trip_window_category",
    "active_hazards",
    "trip_advice"
).show(10, truncate=False)

print("\n⚠️  Worst Times to Travel:")
trip_recommendations.orderBy("trip_suitability_score").select(
    "hour_of_day",
    "time_period",
    "trip_suitability_score",
    "trip_window_category",
    "active_hazards",
    "congestion_level"
).show(5, truncate=False)

print("═" * 80 + "\n")

In [0]:
# Save all outputs to gold layer tables
print("\n" + "═" * 80)
print("💾 SAVING TRIP INTELLIGENCE DATA")
print("═" * 80)

print("\n📝 Writing to gold layer...")

# Save trip routes analysis
trip_routes.write.mode("overwrite").saveAsTable(TRIP_ROUTES_TABLE)
print(f"   ✅ {TRIP_ROUTES_TABLE}: {trip_routes.count():,} routes")

# Save charging requirements
charging_requirements.write.mode("overwrite").saveAsTable(TRIP_CHARGING_TABLE)
print(f"   ✅ {TRIP_CHARGING_TABLE}: {charging_requirements.count():,} distance profiles")

# Save trip recommendations
trip_recommendations.write.mode("overwrite").saveAsTable(TRIP_RECOMMENDATIONS_TABLE)
print(f"   ✅ {TRIP_RECOMMENDATIONS_TABLE}: {trip_recommendations.count():,} time windows")

print("\n" + "─" * 80)
print("\n✅ All trip intelligence tables saved successfully!")
print("═" * 80 + "\n")

In [0]:
print("\n" + "═" * 80)
print("🎯 TRIP INTELLIGENCE ENGINE - EXECUTIVE SUMMARY")
print("═" * 80)

# Load saved tables for summary
routes_tbl = spark.table(TRIP_ROUTES_TABLE)
charging_tbl = spark.table(TRIP_CHARGING_TABLE)
recommendations_tbl = spark.table(TRIP_RECOMMENDATIONS_TABLE)

print("\n📊 Output Summary:")
print(f"   • Route Analysis:      {routes_tbl.count():,} locations analyzed")
print(f"   • Charging Profiles:   {charging_tbl.count():,} distance scenarios")
print(f"   • Time Recommendations: {recommendations_tbl.count():,} windows")

print("\n" + "─" * 80)
print("\n🗺️  Route Safety Overview:")
safest_route = routes_tbl.orderBy("route_risk_score").first()
riskiest_route = routes_tbl.orderBy("route_risk_score", ascending=False).first()

print(f"   Safest Route:")
print(f"   • Location: {safest_route['location'] if safest_route['location'] else 'Multiple safe areas'}")
print(f"   • Risk Score: {safest_route['route_risk_score']}/100")
print(f"   • Active Hazards: {safest_route['active_hazards']}")

print(f"\n   Riskiest Route:")
print(f"   • Location: {riskiest_route['location'] if riskiest_route['location'] else 'High-risk area'}")
print(f"   • Risk Score: {riskiest_route['route_risk_score']}/100")
print(f"   • Active Hazards: {riskiest_route['active_hazards']}")
print(f"   • Expected Delay: {riskiest_route['estimated_delay_minutes']} minutes")

print("\n" + "─" * 80)
print("\n🔋 Charging Infrastructure:")
avg_stops = charging_tbl.agg(F.avg("charging_stops_needed")).first()[0]
max_distance = charging_tbl.agg(F.max("trip_distance_km")).first()[0]
print(f"   • Average Stops Needed: {avg_stops:.1f} per trip")
print(f"   • Maximum Trip Distance Analyzed: {int(max_distance)} km")
print(f"   • Network Coverage: {charging_tbl.first()['charger_availability_score']}")

print("\n" + "─" * 80)
print("\n⏰ Best Travel Windows:")
top_times = recommendations_tbl.orderBy("trip_suitability_score", ascending=False).limit(3).collect()
for i, row in enumerate(top_times, 1):
    print(f"   {i}. {row['hour_of_day']}:00 ({row['time_period']}) - Score: {row['trip_suitability_score']}/100")
    print(f"      → {row['trip_advice']}")

print("\n" + "─" * 80)
print("\n🚫 Times to Avoid:")
worst_times = recommendations_tbl.orderBy("trip_suitability_score").limit(3).collect()
for i, row in enumerate(worst_times, 1):
    print(f"   {i}. {row['hour_of_day']}:00 ({row['time_period']}) - Score: {row['trip_suitability_score']}/100")
    print(f"      → Active hazards: {row['active_hazards']}, Congestion: {row['congestion_level']}")

print("\n" + "─" * 80)
print("\n💡 Key Insights:")

# Calculate statistics
high_risk_routes = routes_tbl.filter(F.col("route_safety_rating").contains("High Risk")).count()
optimal_windows = recommendations_tbl.filter(F.col("trip_window_category") == "Optimal").count()
total_windows = recommendations_tbl.count()

print(f"   • High-Risk Routes to Avoid: {high_risk_routes}")
print(f"   • Optimal Travel Windows: {optimal_windows}/{total_windows} ({optimal_windows/total_windows*100:.1f}%)")
print(f"   • Average Congestion Delay Factor: {recommendations_tbl.agg(F.avg('congestion_delay_factor')).first()[0]:.2f}x")

print("\n" + "─" * 80)
print("\n🎯 Trip Planning Recommendations:")
print("   1. Check route risk scores before departure")
print("   2. Plan charging stops for trips over 300 km")
print("   3. Travel during late night or late morning for best conditions")
print("   4. Avoid evening rush hours (18:00-20:00)")
print("   5. Monitor active hazards in real-time during travel")
print("   6. Ensure 80%+ battery before long trips")
print("   7. Use fast chargers along major routes for efficiency")

print("\n" + "═" * 80)
print("✅ TRIP INTELLIGENCE ENGINE COMPLETE")
print("═" * 80 + "\n")